# Lab 6
Frequency domain filtering using the cameraman image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import data, io
from skimage.transform import resize

In [ ]:
def load_cameraman(target_size=(256, 256)):
    candidates = [
        Path('cameraman.tif'),
        Path('lab6/cameraman.tif'),
        Path('../lab2/cameraman.tif'),
        Path('lab2/cameraman.tif'),
    ]

    for path in candidates:
        if path.exists():
            # Use the local lab image first so the notebook stays self-contained.
            image = io.imread(path)
            break
    else:
        # Fallback to scikit-image sample if the file is not found.
        image = data.camera()

    if image.ndim == 3:
        image = image[..., 0]

    image = resize(image, target_size, anti_aliasing=True)
    return image.astype(np.float64)


def show_images(images, titles, cmap='gray', figsize=(14, 5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles), 1):
        plt.subplot(1, len(images), i)
        plt.imshow(img, cmap=cmap)
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# Task 1
Filtering in the frequency domain using the Laplacian filter.

In [ ]:
image = load_cameraman(target_size=(256, 256))

# Load cameraman image (already grayscale)
# image = data.camera()
# image = resize(image, (256, 256))  # Resize for faster FFT (optional)

# Get image size
M, N = image.shape

# Generate frequency grid (centered)
u = np.fft.fftfreq(M).reshape(-1, 1)
v = np.fft.fftfreq(N).reshape(1, -1)

# Compute Laplacian filter in frequency domain
laplacian_filter = -4 * (np.pi ** 2) * (u ** 2 + v ** 2)

# Apply 2D FFT
F = np.fft.fft2(image)

# Apply Laplacian filter in frequency domain
# Multiplication in frequency domain is equivalent to filtering in space.
F_lap = F * laplacian_filter

# Inverse FFT to get spatial result
# Keep the real part because the small imaginary part comes from numerical error.
laplacian_image = np.fft.ifft2(F_lap).real

# Plot results
show_images(
    [image, np.log1p(np.abs(F_lap)), laplacian_image],
    ['Original Cameraman', 'Laplacian Spectrum', 'Laplacian (Spatial Output)']
)

## Assessment Task 1
Apply Sobel filter(s) in the frequency domain using the same cameraman image.

In [ ]:
# Load and resize image

# Sobel filters
def pad_kernel_centered(kernel, shape):
    # Create an empty array with the same size as the image
    padded = np.zeros(shape)

    # Get kernel height and width
    kh, kw = kernel.shape

    # Get image height and width
    ph, pw = shape

    # Find the center of the image
    cy, cx = ph // 2, pw // 2

    # Place the small kernel in the center of the large padded array
    padded[cy - kh // 2:cy - kh // 2 + kh, cx - kw // 2:cx - kw // 2 + kw] = kernel

    # Shift kernel correctly, then take FFT to move it to frequency domain
    return np.fft.fft2(np.fft.ifftshift(padded))


    # Sobel filter for horizontal edge detection
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=np.float64)

# Sobel filter for vertical edge detection
sobel_y = np.array([[-1, -2, -1],
                    [0, 0, 0],
                    [1, 2, 1]], dtype=np.float64)

# Compute FFT of the image
F = np.fft.fft2(image)

# Convert Sobel kernels to frequency domain
H_x = pad_kernel_centered(sobel_x, image.shape)
H_y = pad_kernel_centered(sobel_y, image.shape)

# Apply Sobel filtering in the frequency domain
G_x = F * H_x
G_y = F * H_y

# Convert filtered results back to spatial domain
sobel_x_image = np.fft.ifft2(G_x).real
sobel_y_image = np.fft.ifft2(G_y).real

# Combine x and y edge results into one magnitude image
sobel_magnitude = np.sqrt(sobel_x_image ** 2 + sobel_y_image ** 2)

# Display original image and Sobel results
show_images(
    [image, sobel_x_image, sobel_y_image, sobel_magnitude],
    ['Original Cameraman', 'Sobel X', 'Sobel Y', 'Sobel Magnitude'],
    figsize=(16, 5)
)